In [38]:
import pandas as pd
import json

In [ ]:
df = pd.read_csv("FAIRe_checklist_v102.csv", sep="\t")
df.head()

,data_type,section,term_name,description,requirement_level_code,requirement_level,requirement_level_condition,term_type,unit,fixed_format,controlled_vocabulary_options,example,sample_type_specificity,source,URI,modifications_made
0,projectMetadata,Project,recordedBy,A list (concatenated and separated) of names o...,M,Mandatory,NaN,free text,NaN,NaN,NaN,José E. Crespo,NaN,DwC,https://dwc.tdwg.org/list/#dwc_recordedBy,description
1,projectMetadata,Project,recordedByID,A list (concatenated and separated) of the glo...,R,Recommended,NaN,free text,NaN,NaN,NaN,https://orcid.org/0000-0002-1825-0097 | http...,NaN,DwC,https://dwc.tdwg.org/list/#dwc_recordedByID,description
2,projectMetadata,Project,project_contact,"A contact person and their contact (i.e., emai...",M,Mandatory,NaN,free text,NaN,NaN,NaN,xxxx@yyyy.com,NaN,New,NaN,NaN
3,projectMetadata,Project,institution,An institution having custody of the object(s)...,R,Recommended,NaN,free text,NaN,NaN,NaN,Curtin University,NaN,New,NaN,NaN
4,projectMetadata,Project,institutionID,An identifier for the institution having custo...,R,Recommended,NaN,free text,NaN,NaN,NaN,https://ror.org/042r9xb17,NaN,DwC,http://rs.tdwg.org/dwc/terms/institutionID,NaN


In [5]:
import pandas as pd

dwc = pd.read_csv("term_versions.csv", sep=",")
dwc = dwc[dwc["status"] == 'recommended']
dwc.shape
dwc.to_csv("dwc_recommded_versions.csv", sep=",", index=False)

In [9]:
from curategpt.store import ChromaDBAdapter

db = ChromaDBAdapter(path="faire_db")

for _, row in df.iterrows():

    obj = {
        "id": f"FAIRe:{row.term_name}",
        "label": row.term_name,
        "description": row.description,
        "syntax": row.fixed_format,
        "datatype": row.term_type,
        "vocabulary": row.controlled_vocabulary_options,
        "example": row.example
    }

    db.insert(obj)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 22050.50it/s]
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


AttributeError: 'str' object has no attribute 'items'

In [11]:
ChromaDBAdapter(path='faire_db', schema_proxy=None, collection=None, _field_names_by_collection=None, default_model='all-MiniLM-L6-v2', client=<chromadb.api.client.Client object at 0x76bc1d0d2600>, id_field='id', text_lookup='text', id_to_object={})

SyntaxError: invalid syntax (2055593514.py, line 1)

In [2]:
dwc = pd.read_csv("/home/camilababo/Documents/coding-projects/curategpt/data/dwc_recommded_versions.csv", sep=',')
dwc.head()

,iri,term_localName,label,definition,comments,examples,organized_in,issued,status,replaces,rdf_type,term_iri,abcd_equivalence,flags
0,http://dublincore.org/usage/terms/history/#typ...,type,Type,The nature or genre of the resource.,Must be populated with a value from the DCMI t...,`StillImage`; `MovingImage`; `Sound`; `Physica...,http://purl.org/dc/elements/1.1/,2025-06-12,recommended,http://dublincore.org/usage/terms/history/#typ...,http://www.w3.org/1999/02/22-rdf-syntax-ns#Pro...,http://purl.org/dc/elements/1.1/type,not in ABCD,simple
1,http://dublincore.org/usage/terms/history/#mod...,modified,Date Modified,Date on which the resource was changed.,Recommended best practice is to use a date tha...,`1963-03-08T14:07-06:00` (8 Mar 1963 at or aft...,http://purl.org/dc/terms/,2025-06-12,recommended,NaN,http://www.w3.org/1999/02/22-rdf-syntax-ns#Pro...,http://purl.org/dc/terms/modified,not in ABCD,simple
2,http://dublincore.org/usage/terms/history/#lan...,language,Language,A language of the resource.,Recommended best practice is to use a controll...,`en` (for English); `es` (for Spanish),http://purl.org/dc/elements/1.1/,2008-01-14,recommended,http://dublincore.org/usage/terms/history/#lan...,http://www.w3.org/1999/02/22-rdf-syntax-ns#Pro...,http://purl.org/dc/elements/1.1/language,not in ABCD,simple
3,http://dublincore.org/usage/terms/history/#lic...,license,License,A legal document giving official permission to...,NaN,`http://creativecommons.org/publicdomain/zero/...,http://purl.org/dc/terms/,2008-01-14,recommended,http://dublincore.org/usage/terms/history/#rig...,http://www.w3.org/1999/02/22-rdf-syntax-ns#Pro...,http://purl.org/dc/terms/license,not in ABCD,simple
4,http://dublincore.org/usage/terms/history/#rig...,rightsHolder,Rights Holder,A person or organization owning or managing ri...,NaN,`The Regents of the University of California`,http://purl.org/dc/terms/,2008-01-14,recommended,NaN,http://www.w3.org/1999/02/22-rdf-syntax-ns#Pro...,http://purl.org/dc/terms/rightsHolder,not in ABCD,simple


In [36]:
def dwc_dict(df):
    dict = {}

    for _, row in df.iterrows():
        value = row['examples'].split(";")[0] if pd.notna(row["examples"]) else ""
        dict[row["term_localName"]] = {
            "value": value.replace("´", "").replace("`", ""), 
            "description": row["definition"]
            }
        
    return dict

dwc_dict = dwc_dict(dwc)

In [42]:
dwc_dict

with open("dwc_term_history_filt.json", "w") as f:
    json.dump(dwc_dict, f, indent=2)